# 🚀 Quranic Qira'at ML - Training Guide

Complete training walkthrough with monitoring and checkpointing.

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))

import torch
from torch.utils.data import DataLoader
from datasets import load_from_disk
from transformers import Wav2Vec2Processor
import wandb
import numpy as np
from tqdm import tqdm

from src.train import (
    QiraatMTLModel,
    TrainingConfig,
    QiraatTrainer,
    compute_mtl_loss,
)
from src.preprocess import QuranicAudioDataset

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## Configuration

In [ ]:
# Create training config
config = TrainingConfig(
    model_name="facebook/wav2vec2-xlsr-128d",
    lora_rank=8,
    lora_alpha=32,
    
    # Training parameters
    epochs=15,
    batch_size=4,
    accumulation_steps=4,
    learning_rate=5e-4,
    warmup_steps=500,
    
    # Loss weighting
    weight_transcription=0.5,
    weight_qiraat=0.3,
    weight_tajweed=0.2,
    
    # Optimization
    use_mixed_precision=True,
    use_gradient_checkpointing=True,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print("⚙️  Training Configuration:")
print(f"  Epochs: {config.epochs}")
print(f"  Batch size: {config.batch_size}")
print(f"  Learning rate: {config.learning_rate}")
print(f"  Mixed precision: {config.use_mixed_precision}")
print(f"  Gradient checkpointing: {config.use_gradient_checkpointing}")
print(f"  Device: {config.device}")

## Load Data

In [ ]:
# Load preprocessed dataset
print("📂 Loading dataset...")
try:
    dataset_dict = load_from_disk('./data/cache/dataset_splits')
    print(f"✅ Loaded dataset splits")
    print(f"  Train: {len(dataset_dict['train'])} samples")
    print(f"  Validation: {len(dataset_dict['validation'])} samples")
    print(f"  Test: {len(dataset_dict['test'])} samples")
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    print("Run preprocess.py first to prepare the dataset")

## Initialize Model & Trainer

In [ ]:
print("🔧 Initializing model...")

# Create model
model = QiraatMTLModel(
    model_name=config.model_name,
    lora_rank=config.lora_rank,
    lora_alpha=config.lora_alpha,
    use_gradient_checkpointing=config.use_gradient_checkpointing,
)

# Create trainer
trainer = QiraatTrainer(model, config)

print(f"✅ Model initialized")
print(f"  Device: {trainer.device}")
print(f"  Mixed precision: {config.use_mixed_precision}")

## Check VRAM

In [ ]:
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f"💾 VRAM Status:")
    print(f"  Allocated: {allocated:.2f}GB / {total:.2f}GB")
    print(f"  Reserved: {reserved:.2f}GB")
    print(f"  Available: {total - allocated:.2f}GB")
    
    usage_pct = (allocated / total) * 100
    status = "✅" if usage_pct < 80 else "⚠️"
    print(f"\n{status} Usage: {usage_pct:.1f}%")

## Setup Weights & Biases (Optional)

In [ ]:
# Initialize wandb for tracking
try:
    wandb.init(
        project="quranic-qiraat-ml",
        name="training_session",
        config={
            'epochs': config.epochs,
            'batch_size': config.batch_size,
            'learning_rate': config.learning_rate,
        }
    )
    print("✅ Weights & Biases initialized")
except Exception as e:
    print(f"⚠️  W&B not available: {e}")
    print("Training will continue without experiment tracking")

## Train Model (Single Epoch Example)

In [ ]:
# For demonstration, train on small subset
print("🚀 Starting training (single epoch for demo)...\n")

from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Create small dataloader for testing
# (In production, use the full dataset)
train_subset = dataset_dict['train'].select(range(min(10, len(dataset_dict['train']))))

# Note: Full training would use:
# processor = Wav2Vec2Processor.from_pretrained(config.model_name)
# train_dataset = QuranicAudioDataset(
#     dataset_dict['train'],
#     processor,
#     qiraat_to_id={'Hafs': 0, 'Warsh': 1},
# )
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=config.batch_size,
#     shuffle=True,
# )

print("✅ Training setup complete")
print("\n📌 For full training, run: python src/train.py")

## Monitor Training Progress

In [ ]:
import matplotlib.pyplot as plt

# Example: Visualize expected training curves
epochs = np.arange(1, 16)
example_train_loss = 0.5 * np.exp(-epochs / 3) + 0.1
example_val_loss = 0.6 * np.exp(-epochs / 3) + 0.12

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(epochs, example_train_loss, 'b-', label='Train Loss', marker='o')
axes[0].plot(epochs, example_val_loss, 'r-', label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Expected Training Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
qiraat_accuracy = 70 + 20 * (1 - np.exp(-epochs / 2))
axes[1].plot(epochs, qiraat_accuracy, 'g-', label='Qira\'at Accuracy', marker='o')
axes[1].axhline(y=92, color='k', linestyle='--', alpha=0.5, label='Target (92%)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Expected Accuracy Progression')
axes[1].set_ylim([60, 100])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Expected results after 15 epochs:")
print(f"  Qira'at Accuracy: 92-95%")
print(f"  Final Train Loss: ~0.15")
print(f"  Final Val Loss: ~0.18")

## Training Timeline

In [ ]:
import pandas as pd

timeline = pd.DataFrame({
    'Phase': ['Warm-up', 'Fine-tuning', 'Polish'],
    'Epochs': ['1-2', '3-10', '11-15'],
    'Duration (approx)': ['1.5h', '6h', '3.5h'],
    'Strategy': [
        'Freeze encoder, train LoRA',
        'Unfreeze LoRA, train all',
        'Low LR, monitor metrics'
    ],
    'Learning Rate': ['1e-4', '5e-5', '1e-5'],
})

print("⏱️  Training Timeline (RTX 5070 Ti, batch_size=4):\n")
print(timeline.to_string(index=False))
print(f"\nTotal Time: ~12-15 hours")

## Next Steps

1. **Run full training**: `python src/train.py`
2. **Monitor W&B dashboard**: https://wandb.ai/your-username/quranic-qiraat-ml
3. **Check logs**: `tail -f logs/training.log`
4. **Evaluate checkpoints**: See `3_evaluation.ipynb`